# GemmaFit v5 E2B Evidence Router Training

Colab-resumable notebook for the v5 E2B evidence router. The model learns compact evidence packet -> one bounded function call. It does not learn raw video, raw skeleton math, force, GRF, EMG, heart-rate status, fall-risk prediction, clinical labels, or medical conclusions.


In [1]:
# Phase -1 / 0: Resume diagnostic, Drive mount, paths, and atomic state helpers
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Training preset. Default is a repair smoke run for the hardened generator.
# Set GEMMAFIT_TRAIN_PRESET=full before this cell to run the full 50k / 6250-step clean run.
GEMMAFIT_TRAIN_PRESET = os.environ.get('GEMMAFIT_TRAIN_PRESET', 'smoke').strip().lower()
if GEMMAFIT_TRAIN_PRESET in {'smoke', 'repair_smoke', 'smoke_400', 'cheap'}:
    os.environ.setdefault('RUN_SUFFIX_OVERRIDE', 'gemmafit_v5_2_e2b_completion_only_repair3_contract_smoke')
    os.environ.setdefault('FORCE_FRESH_TEXT_LORA_RUN', '1')
    os.environ.setdefault('TRAIN_LIMIT', '6400')
    os.environ.setdefault('EVAL_LIMIT', '256')
    os.environ.setdefault('NUM_TRAIN_EPOCHS', '1')
    os.environ.setdefault('MAX_SEQ_LENGTH', '3072')
    os.environ.setdefault('MODEL_EVAL_MAX_NEW_TOKENS', '768')
    os.environ.setdefault('DATASET_SCHEMA_FUZZ_RATIO', '0.18')
    os.environ.setdefault('DATASET_TOOL_SCHEMA_HARDENING_RATIO', '0.25')
elif GEMMAFIT_TRAIN_PRESET in {'full', 'full_6250', 'clean_full'}:
    os.environ.setdefault('RUN_SUFFIX_OVERRIDE', 'gemmafit_v5_2_e2b_completion_only_repair_contract_v5')
    os.environ.setdefault('FORCE_FRESH_TEXT_LORA_RUN', '1')
    os.environ.pop('TRAIN_LIMIT', None)
    os.environ.setdefault('EVAL_LIMIT', '256')
    os.environ.setdefault('NUM_TRAIN_EPOCHS', '1')
    os.environ.setdefault('MAX_SEQ_LENGTH', '3072')
    os.environ.setdefault('MODEL_EVAL_MAX_NEW_TOKENS', '768')
    os.environ.setdefault('DATASET_SCHEMA_FUZZ_RATIO', '0.18')
    os.environ.setdefault('DATASET_TOOL_SCHEMA_HARDENING_RATIO', '0.25')
else:
    raise ValueError(f'Unknown GEMMAFIT_TRAIN_PRESET={GEMMAFIT_TRAIN_PRESET!r}; use smoke or full')
print('Training preset:', GEMMAFIT_TRAIN_PRESET)
print('RUN_SUFFIX_OVERRIDE:', os.environ.get('RUN_SUFFIX_OVERRIDE'))
print('FORCE_FRESH_TEXT_LORA_RUN:', os.environ.get('FORCE_FRESH_TEXT_LORA_RUN'))
print('TRAIN_LIMIT:', os.environ.get('TRAIN_LIMIT', '<full dataset>'))
print('EVAL_LIMIT:', os.environ.get('EVAL_LIMIT'))
print('NUM_TRAIN_EPOCHS:', os.environ.get('NUM_TRAIN_EPOCHS'))
print('DATASET_SCHEMA_FUZZ_RATIO:', os.environ.get('DATASET_SCHEMA_FUZZ_RATIO'))
print('DATASET_TOOL_SCHEMA_HARDENING_RATIO:', os.environ.get('DATASET_TOOL_SCHEMA_HARDENING_RATIO'))

# Overnight safety: after train/merge, run the real generative tool-call eval and then disconnect.
os.environ.setdefault('RUN_GENERATION_EVAL_AFTER_TRAIN', '1')
os.environ.setdefault('GEN_EVAL_LIMIT', '320')
os.environ.setdefault('GEN_EVAL_SEED', '42')
os.environ.setdefault('GEN_EVAL_MAX_NEW_TOKENS', os.environ.get('MODEL_EVAL_MAX_NEW_TOKENS', '768'))
os.environ.setdefault('GEN_EVAL_PROGRESS_EVERY', '10')
os.environ.setdefault('GEN_EVAL_STRICT', '0')
os.environ.setdefault('AUTO_DISCONNECT_AFTER_DONE', '1')
print('RUN_GENERATION_EVAL_AFTER_TRAIN:', os.environ.get('RUN_GENERATION_EVAL_AFTER_TRAIN'))
print('GEN_EVAL_LIMIT:', os.environ.get('GEN_EVAL_LIMIT'))
print('GEN_EVAL_SEED:', os.environ.get('GEN_EVAL_SEED'))
print('GEN_EVAL_MAX_NEW_TOKENS:', os.environ.get('GEN_EVAL_MAX_NEW_TOKENS'))
print('GEN_EVAL_PROGRESS_EVERY:', os.environ.get('GEN_EVAL_PROGRESS_EVERY'))
print('GEN_EVAL_STRICT:', os.environ.get('GEN_EVAL_STRICT'))
print('AUTO_DISCONNECT_AFTER_DONE:', os.environ.get('AUTO_DISCONNECT_AFTER_DONE'))

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive_mount_point = Path('/content/drive')
    if (drive_mount_point / 'MyDrive').exists():
        print('Drive already mounted at /content/drive; skipping mount.')
    else:
        try:
            drive.mount('/content/drive')
        except ValueError as exc:
            if 'Mountpoint must not already contain files' in str(exc):
                alt_drive_mount_point = Path('/content/gdrive')
                drive.mount(str(alt_drive_mount_point))
                os.environ.setdefault('GEMMAFIT_DRIVE_ROOT', str(alt_drive_mount_point / 'MyDrive' / 'GemmaFit_train'))
                print('Mounted Drive at /content/gdrive and set GEMMAFIT_DRIVE_ROOT.')
            else:
                raise

DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
GEMMAFIT_GIT_REF = os.environ.get('GEMMAFIT_GIT_REF', 'codex/e2b-video-capability-probes')
DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_2_hard')
USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'v5_2', 'v5_2_hard', 'hard'}
RUN_SUFFIX = ('gemmafit_v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router')
RUN_SUFFIX = os.environ.get('RUN_SUFFIX_OVERRIDE', RUN_SUFFIX)
BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
TRAINED = DRIVE_ROOT / 'trained_outputs'
CKPT_DIR = TRAINED / 'checkpoints' / RUN_NAME
ADAPTER_DIR = TRAINED / 'adapter' / RUN_NAME
MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
PHASE_DIR = TRAINED / f'PHASE_DONE_{RUN_NAME}'
RUN_STATE = TRAINED / f'RUN_STATE_{RUN_NAME}.json'
RUN_EVENTS = TRAINED / f'RUN_EVENTS_{RUN_NAME}.jsonl'
DISCONNECT_POINTS = TRAINED / f'DISCONNECT_POINTS_{RUN_NAME}.jsonl'
DONE_MARKER = TRAINED / f'TRAINING_DONE_{RUN_NAME}.json'
DATASET_REPO_PATH = Path('finetune/data/gemmafit_v5_2_e2b_evidence_router.json' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'finetune/data/gemmafit_v5_1_e2b_evidence_router.json' if USE_HARD_CASES else 'finetune/data/gemmafit_v5_e2b_evidence_router.json')
DATASET_TRAIN_COUNT = int(os.environ.get('DATASET_TRAIN_COUNT', '50000' if USE_HARD_CASES else '8000'))
DATASET_VALIDATION_COUNT = int(os.environ.get('DATASET_VALIDATION_COUNT', '6000' if USE_HARD_CASES else '1200'))
DATASET_ZH_TW_RATIO = float(os.environ.get('DATASET_ZH_TW_RATIO', '0.45' if USE_HARD_CASES else '0.0'))
DATASET_SCHEMA_FUZZ_RATIO = float(os.environ.get('DATASET_SCHEMA_FUZZ_RATIO', '0.18')) if USE_HARD_CASES else None
DATASET_TOOL_SCHEMA_HARDENING_RATIO = float(os.environ.get('DATASET_TOOL_SCHEMA_HARDENING_RATIO', '0.25')) if USE_HARD_CASES else None
DATASET_PATH = REPO_DIR / DATASET_REPO_PATH
METRICS_DIR = REPO_DIR / 'finetune' / 'metrics'
TOOL_EVAL = METRICS_DIR / 'tool_call_eval_v5_e2b.json'
REFUSAL_EVAL = METRICS_DIR / 'refusal_eval_v5_e2b.json'
ADVERSARIAL_EVAL = METRICS_DIR / 'adversarial_eval_v5_e2b.json'

for p in [TRAINED, CKPT_DIR, ADAPTER_DIR, MERGED_DIR, PHASE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def now_iso() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    tmp.replace(path)

def record_event(phase: str, event: str, **kwargs) -> None:
    RUN_EVENTS.parent.mkdir(parents=True, exist_ok=True)
    payload = {'ts': now_iso(), 'run_name': RUN_NAME, 'phase': phase, 'event': event, **kwargs}
    with RUN_EVENTS.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
    if event in {'dataset_validated', 'trainer_initialized', 'checkpoint_saved', 'adapter_saved', 'merged_hf_saved', 'eval_done', 'done_marker_written'}:
        with DISCONNECT_POINTS.open('a', encoding='utf-8') as f:
            f.write(json.dumps({**payload, 'note': 'safe_resume_point'}, ensure_ascii=False) + '\n')

def write_run_state(phase: str, **kwargs) -> None:
    state = {'version': 'v5_e2b_evidence_router', 'run_name': RUN_NAME, 'run_suffix': RUN_SUFFIX, 'phase': phase, 'updated_at': now_iso(), **kwargs}
    atomic_json(RUN_STATE, state)

def mark_phase_done(phase_id: str, outputs: list[str] | None = None, next_phase: str = '') -> None:
    payload = {'phase': phase_id, 'status': 'done', 'timestamp': now_iso(), 'run_name': RUN_NAME, 'outputs': outputs or [], 'next_phase': next_phase}
    atomic_json(PHASE_DIR / f'{phase_id}.json', payload)
    write_run_state(phase_id, last_completed_phase=phase_id, next_phase=next_phase)

def latest_checkpoint(path: Path) -> str | None:
    checkpoints = []
    if path.exists():
        for child in path.glob('checkpoint-*'):
            if child.is_dir():
                try:
                    checkpoints.append((int(child.name.split('-')[-1]), child))
                except ValueError:
                    pass
    return str(sorted(checkpoints)[-1][1]) if checkpoints else None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

print('=== Resume diagnostic ===')
print('Run name:', RUN_NAME)
print('Base model:', BASE_MODEL_ID)
print('Done marker:', DONE_MARKER, DONE_MARKER.exists())
print('Last completed phase:', sorted([p.stem for p in PHASE_DIR.glob('*.json')])[-1:] or 'none')
print('Latest checkpoint:', latest_checkpoint(CKPT_DIR))
print('Repo dir:', REPO_DIR)
print('Dataset path:', DATASET_PATH)
print('Artifacts found:', [str(p) for p in [RUN_STATE, RUN_EVENTS, DONE_MARKER] if p.exists()])
record_event('0_paths', 'diagnostic_printed')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Resume diagnostic ===
Run name: google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases
Base model: google/gemma-4-E2B-it
Done marker: /content/drive/MyDrive/GemmaFit_train/trained_outputs/TRAINING_DONE_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.json False
Last completed phase: ['4_model_preflight']
Latest checkpoint: None
Repo dir: /content/GemmaFit
Dataset path: /content/GemmaFit/finetune/data/gemmafit_v5_1_e2b_evidence_router.json
Artifacts found: ['/content/drive/MyDrive/GemmaFit_train/trained_outputs/RUN_STATE_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.json', '/content/drive/MyDrive/GemmaFit_train/trained_outputs/RUN_EVENTS_google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases.jsonl']


In [2]:
# Phase 1: Install dependencies and load HF token
# Set INSTALL_DEPS=0 to skip when the runtime already has compatible packages.
INSTALL_DEPS = os.environ.get('INSTALL_DEPS', '1') == '1'
if INSTALL_DEPS:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U',
        'transformers', 'datasets', 'accelerate', 'peft', 'trl', 'safetensors', 'huggingface_hub'
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping dependency install because INSTALL_DEPS=0')

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('HF token loaded from environment')
    except Exception as exc:
        print('HF login skipped/failed:', exc)
else:
    print('No HF_TOKEN in environment. Public models may still load; gated models will fail.')
mark_phase_done('1_deps', ['transformers', 'datasets', 'peft', 'trl'], '2_repo_dataset')


Running: /usr/bin/python3 -m pip install -U transformers datasets accelerate peft trl safetensors huggingface_hub
No HF_TOKEN in environment. Public models may still load; gated models will fail.


In [3]:
# Phase 2 / 3: Repo checkout, v5 dataset generation, strict schema gates
REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')

def run_repo_git(args: list[str], *, check: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(args, cwd=str(REPO_DIR), text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip(), file=sys.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, args, output=result.stdout, stderr=result.stderr)
    return result

def clone_repo_fresh() -> None:
    if REPO_DIR.exists():
        resolved_repo = REPO_DIR.resolve()
        if str(resolved_repo).startswith('/content/') and resolved_repo.name == 'GemmaFit':
            print('Removing stale Colab checkout:', resolved_repo)
            shutil.rmtree(resolved_repo)
        else:
            raise RuntimeError(f'Refusing to remove non-Colab repo dir: {resolved_repo}')
    clone_cmd = ['git', 'clone']
    if GEMMAFIT_GIT_REF:
        clone_cmd.extend(['--branch', GEMMAFIT_GIT_REF, '--single-branch'])
    clone_cmd.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(clone_cmd, check=True)

if not REPO_DIR.exists():
    clone_repo_fresh()
else:
    print('Using existing repo:', REPO_DIR)

if GEMMAFIT_GIT_REF:
    print('Checking out repo ref:', GEMMAFIT_GIT_REF)
    fetch_result = run_repo_git(['git', 'fetch', 'origin', GEMMAFIT_GIT_REF], check=False)
    if fetch_result.returncode != 0:
        print('Git fetch failed; recloning the ephemeral Colab checkout.')
        clone_repo_fresh()
    else:
        run_repo_git(['git', 'checkout', '-B', GEMMAFIT_GIT_REF, f'origin/{GEMMAFIT_GIT_REF}'])
        run_repo_git(['git', 'pull', '--ff-only', 'origin', GEMMAFIT_GIT_REF])
else:
    print('GEMMAFIT_GIT_REF is empty; using repo default checkout.')

os.chdir(REPO_DIR)
subprocess.run([sys.executable, 'finetune/data/generate_v5_e2b_evidence_router.py', '--help'], check=True)
generate_cmd = [
    sys.executable,
    'finetune/data/generate_v5_e2b_evidence_router.py',
    '--train-count', str(DATASET_TRAIN_COUNT),
    '--validation-count', str(DATASET_VALIDATION_COUNT),
    '--out', str(DATASET_REPO_PATH),
    '--validate',
]
if USE_HARD_CASES:
    generate_cmd.extend(['--hard-cases', '--zh-tw-ratio', str(DATASET_ZH_TW_RATIO)])
    if DATASET_VARIANT in {'v5_2', 'v5_2_hard'}:
        generate_cmd.append('--tool-contract-v2')
    if DATASET_SCHEMA_FUZZ_RATIO is not None:
        generate_cmd.extend(['--schema-fuzz-ratio', str(DATASET_SCHEMA_FUZZ_RATIO)])
    if DATASET_TOOL_SCHEMA_HARDENING_RATIO is not None:
        generate_cmd.extend(['--tool-schema-hardening-ratio', str(DATASET_TOOL_SCHEMA_HARDENING_RATIO)])
print('Dataset generation command:', ' '.join(generate_cmd))
subprocess.run(generate_cmd, check=True)
subprocess.run([
    sys.executable,
    'finetune/eval_v5_e2b_evidence_router.py',
    '--dataset', str(DATASET_REPO_PATH),
    '--strict',
], check=True)
DATASET_SHA256 = sha256_file(DATASET_PATH)
with DATASET_PATH.open('r', encoding='utf-8') as f:
    dataset_payload = json.load(f)
TRAIN_ROWS = len(dataset_payload['train'])
VALIDATION_ROWS = sum(len(v) for v in dataset_payload['validation'].values())
print({'dataset_sha256': DATASET_SHA256, 'train_rows': TRAIN_ROWS, 'validation_rows': VALIDATION_ROWS})
record_event('3_dataset', 'dataset_validated', dataset=str(DATASET_PATH), sha256=DATASET_SHA256, train_rows=TRAIN_ROWS, validation_rows=VALIDATION_ROWS)
mark_phase_done('3_dataset_validated', [str(DATASET_PATH), str(TOOL_EVAL)], '4_model_preflight')


Using existing repo: /content/GemmaFit
Checking out repo ref: codex/e2b-video-capability-probes
Dataset generation command: /usr/bin/python3 finetune/data/generate_v5_e2b_evidence_router.py --train-count 50000 --validation-count 6000 --out finetune/data/gemmafit_v5_1_e2b_evidence_router.json --validate --hard-cases --zh-tw-ratio 0.45 --schema-fuzz-ratio 0.25
{'dataset_sha256': '14275d70f1d15927cb1b1f7ae1bf19677fd155b5e9c5601fd5909c0d8d8f4ddc', 'train_rows': 50000, 'validation_rows': 6000}


In [4]:
# Phase 4: Model/tokenizer preflight and dataset preview
from datasets import Dataset
from transformers import AutoTokenizer

with DATASET_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

print('Validation splits:', {k: len(v) for k, v in data['validation'].items()})
print('First row type:', data['train'][0].get('row_type'))
print('First assistant target:', data['train'][0]['messages'][-1]['content'][:500])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_messages(row: dict) -> str:
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
    return '\n'.join(f"{m['role']}: {m['content']}" for m in row['messages'])

def render_prompt_for_training(row: dict) -> str:
    prompt_messages = row['messages'][:-1]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    return '\n'.join(f"{m['role']}: {m['content']}" for m in prompt_messages) + '\nassistant: '

def assistant_target(row: dict) -> str:
    return row['messages'][-1]['content']

MAX_SEQ_LENGTH_PREVIEW = int(os.environ.get('MAX_SEQ_LENGTH', '3072'))
TOKEN_AUDIT_TRAIN_LIMIT = int(os.environ.get('TOKEN_AUDIT_TRAIN_LIMIT', '2048'))
TOKEN_AUDIT_VALIDATION_PER_SPLIT = int(os.environ.get('TOKEN_AUDIT_VALIDATION_PER_SPLIT', '32'))
audit_rows = list(data['train'][:TOKEN_AUDIT_TRAIN_LIMIT])
for split_rows in data['validation'].values():
    audit_rows.extend(split_rows[:TOKEN_AUDIT_VALIDATION_PER_SPLIT])

full_lengths = []
prompt_lengths = []
target_lengths = []
overflows = []
for idx, row in enumerate(audit_rows):
    prompt_ids = tokenizer(render_prompt_for_training(row), add_special_tokens=False).input_ids
    target_ids = tokenizer(assistant_target(row), add_special_tokens=False).input_ids
    full_len = len(prompt_ids) + len(target_ids)
    prompt_lengths.append(len(prompt_ids))
    target_lengths.append(len(target_ids))
    full_lengths.append(full_len)
    if len(prompt_ids) >= MAX_SEQ_LENGTH_PREVIEW or full_len > MAX_SEQ_LENGTH_PREVIEW:
        overflows.append({
            'audit_index': idx,
            'row_type': row.get('row_type'),
            'expected_function': row.get('expected_function'),
            'prompt_tokens': len(prompt_ids),
            'target_tokens': len(target_ids),
            'total_tokens': full_len,
        })

token_report = {
    'rows_checked': len(audit_rows),
    'max_seq_length': MAX_SEQ_LENGTH_PREVIEW,
    'prompt_tokens_max': max(prompt_lengths),
    'target_tokens_max': max(target_lengths),
    'total_tokens_max': max(full_lengths),
    'possible_target_truncation_rows': len(overflows),
    'overflow_sample': overflows[:5],
}
print('TOKEN_AUDIT', json.dumps(token_report, indent=2, ensure_ascii=False))
if overflows:
    raise RuntimeError(f'Token audit found rows that can truncate assistant targets: {overflows[:3]}')
mark_phase_done('4_model_preflight', [BASE_MODEL_ID], '5_lora_setup')


Validation splits: {'care_log': 400, 'persona_report': 400, 'dual_task': 400, 'realtime_event': 400, 'activity_uncertain': 400, 'unsupported': 400, 'adversarial': 400, 'zh_tw': 400, 'schema_fuzz': 400, 'tracking_uncertainty': 400, 'parent_task_uncertain': 400, 'sub_action_fallback': 400, 'conflicting_evidence': 400, 'memory_policy': 400, 'unsupported_zh_tw': 400}
First row type: tracking_uncertainty
First assistant target: {"function":"refuse_unsupported_question","args":{"reason":"insufficient_evidence","safe_alternative":"Tracking is predicted for this window, so hard coaching is blocked. Please use a clearer single-person view before feedback.","selection_basis":"Person tracking does not allow hard judgment for this event packet.","evidence_refs":["tracking.judgment.blocked"],"refusal_level":3}}
{'max_sample_tokens': 1505, 'avg_sample_tokens': 1198.6328125}


In [5]:
import subprocess
import sys
import os
# Phase 5 / 6: LoRA SFT training with checkpoint resume
# Fix: Install bitsandbytes if not already installed.
try:
    import bitsandbytes
except ImportError:
    print("bitsandbytes not found, installing...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'bitsandbytes>=0.46.1'], check=True)
    print("bitsandbytes installed.")
    import bitsandbytes

import importlib
importlib.invalidate_caches()
if 'transformers' in sys.modules:
    import transformers.utils.import_utils
    transformers.utils.import_utils._bitsandbytes_available = True
    print("Forced transformers to recognize bitsandbytes.")

from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
import torch
import inspect
import json

MAX_SEQ_LENGTH = int(os.environ.get('MAX_SEQ_LENGTH', '3072'))
TRAIN_LIMIT = int(os.environ.get('TRAIN_LIMIT', '0'))
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '256'))
OUTPUT_DIR = str(CKPT_DIR)

train_rows = data['train'][:TRAIN_LIMIT or None]
eval_pool = []
for split_rows in data['validation'].values():
    eval_pool.extend(split_rows)
eval_rows = eval_pool[:EVAL_LIMIT]

def render_prompt(row: dict) -> str:
    prompt_messages = row['messages'][:-1]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    return '\n'.join(f"{m['role']}: {m['content']}" for m in prompt_messages) + '\nassistant: '

def assistant_completion(row: dict) -> str:
    target = row['messages'][-1]['content']
    json.loads(target)
    forbidden = ['```', 'activity_context', 'person_tracking_state', 'motion_feature_window', 'evidence_ledger', 'capability_contract']
    leaked = [name for name in forbidden if name in target]
    if leaked:
        raise ValueError(f'Assistant target leaked input sections or fences: {leaked}')
    return target

def to_prompt_completion_dataset(rows: list[dict]) -> Dataset:
    return Dataset.from_list([
        {'prompt': render_prompt(row), 'completion': assistant_completion(row), 'row_type': row.get('row_type', '')}
        for row in rows
    ])

train_dataset = to_prompt_completion_dataset(train_rows)
eval_dataset = to_prompt_completion_dataset(eval_rows)

use_4bit = os.environ.get('LOAD_IN_4BIT', '1') == '1'
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type='nf4') if use_4bit else None
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
)
if use_4bit:
    model = prepare_model_for_kbit_training(model)

BASE_LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LORA_TEXT_MARKERS = ('.language_model.', '.text_model.', '.decoder.')
LORA_EXCLUDED_MARKERS = ('.vision_tower.', '.audio_tower.')

def resolve_lora_target_modules(model, base_targets: list[str]) -> list[str]:
    module_names = [name for name, _ in model.named_modules()]
    resolved: list[str] = []
    for name in module_names:
        if any(marker in name for marker in LORA_EXCLUDED_MARKERS):
            continue
        if not any(marker in name for marker in LORA_TEXT_MARKERS):
            continue
        if any(name.endswith(f'.{target}') for target in base_targets):
            resolved.append(name)
    found_suffixes = {name.rsplit('.', 1)[-1] for name in resolved}
    missing = sorted(set(base_targets) - found_suffixes)
    if missing:
        raise RuntimeError(f'Missing text LoRA target suffixes: {missing}. Inspect model.named_modules().')
    if not resolved:
        raise RuntimeError('No text decoder LoRA target modules found. Refusing to target vision/audio towers.')
    blocked = [name for name in resolved if any(marker in name for marker in LORA_EXCLUDED_MARKERS)]
    if blocked:
        raise RuntimeError(f'LoRA target resolver selected non-text towers: {blocked[:8]}')
    return resolved

def lora_b_nonzero_stats(model) -> dict:
    text_b = []
    blocked_b = []
    for name, param in model.named_parameters():
        if 'lora_B' not in name:
            continue
        is_text = any(marker.strip('.') in name for marker in LORA_TEXT_MARKERS)
        is_blocked = any(marker.strip('.') in name for marker in LORA_EXCLUDED_MARKERS)
        if is_blocked:
            blocked_b.append((name, int((param.detach() != 0).sum().item()), int(param.numel())))
        elif is_text:
            text_b.append((name, int((param.detach() != 0).sum().item()), int(param.numel())))
    return {
        'text_lora_B_tensors': len(text_b),
        'text_lora_B_nonzero_tensors': sum(1 for _, nonzero, _ in text_b if nonzero > 0),
        'text_lora_B_nonzero_params': sum(nonzero for _, nonzero, _ in text_b),
        'blocked_lora_B_tensors': len(blocked_b),
        'blocked_lora_B_nonzero_tensors': sum(1 for _, nonzero, _ in blocked_b if nonzero > 0),
    }

lora_target_modules = resolve_lora_target_modules(model, BASE_LORA_TARGETS)
print('LoRA target module count:', len(lora_target_modules))
print('LoRA target module preview:', lora_target_modules[:14])
record_event('6_training', 'lora_targets_resolved', target_modules=lora_target_modules)

lora_config = LoraConfig(
    r=int(os.environ.get('LORA_R', '16')),
    lora_alpha=int(os.environ.get('LORA_ALPHA', '32')),
    lora_dropout=float(os.environ.get('LORA_DROPOUT', '0.05')),
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=lora_target_modules,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
pre_train_lora_stats = lora_b_nonzero_stats(model)
print('Pre-train LoRA B stats:', pre_train_lora_stats)
if pre_train_lora_stats['blocked_lora_B_tensors']:
    raise RuntimeError(f'LoRA attached to blocked towers: {pre_train_lora_stats}')
if pre_train_lora_stats['text_lora_B_tensors'] == 0:
    raise RuntimeError('LoRA did not attach to the text decoder.')
record_event('6_training', 'trainer_initialized', checkpoint_dir=str(CKPT_DIR))

def build_sft_config() -> SFTConfig:
    signature = inspect.signature(SFTConfig.__init__)
    params = signature.parameters
    accepts_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
    bf16_ok = bool(torch.cuda.is_available() and getattr(torch.cuda, 'is_bf16_supported', lambda: True)())
    requested = {
        'output_dir': OUTPUT_DIR,
        'per_device_train_batch_size': int(os.environ.get('BATCH_SIZE', '1')),
        'gradient_accumulation_steps': int(os.environ.get('GRAD_ACCUM', '8')),
        'learning_rate': float(os.environ.get('LEARNING_RATE', '2e-4')),
        'num_train_epochs': float(os.environ.get('NUM_TRAIN_EPOCHS', '1')),
        'logging_steps': 10,
        'save_strategy': 'steps',
        'save_steps': 50,
        'save_total_limit': 3,
        'eval_steps': 50,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'eval_loss',
        'bf16': bf16_ok,
        'report_to': [],
    }
    if 'max_seq_length' in params:
        requested['max_seq_length'] = MAX_SEQ_LENGTH
    elif 'max_length' in params:
        requested['max_length'] = MAX_SEQ_LENGTH
    else:
        print('SFTConfig has no max_seq_length/max_length parameter; using tokenizer defaults.')
    if 'eval_strategy' in params:
        requested['eval_strategy'] = 'steps'
    elif 'evaluation_strategy' in params:
        requested['evaluation_strategy'] = 'steps'
    if 'completion_only_loss' in params:
        requested['completion_only_loss'] = True
    elif 'assistant_only_loss' in params:
        requested['assistant_only_loss'] = True
    else:
        raise RuntimeError('This TRL version does not expose completion_only_loss/assistant_only_loss; refusing full-prompt SFT on long evidence.')
    filtered = requested if accepts_kwargs else {k: v for k, v in requested.items() if k in params}
    dropped = sorted(set(requested) - set(filtered))
    if dropped:
        print('Dropped unsupported SFTConfig args:', dropped)
    print('SFTConfig length arg:', 'max_seq_length' if 'max_seq_length' in filtered else ('max_length' if 'max_length' in filtered else 'default'))
    return SFTConfig(**filtered)

args = build_sft_config()

trainer_signature = inspect.signature(SFTTrainer.__init__)
trainer_params = trainer_signature.parameters
trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': train_dataset,
    'eval_dataset': eval_dataset,
}
if 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
if 'max_seq_length' in trainer_params:
    trainer_kwargs['max_seq_length'] = MAX_SEQ_LENGTH
trainer = SFTTrainer(**trainer_kwargs)
resume = latest_checkpoint(CKPT_DIR)
if os.environ.get('FORCE_FRESH_TEXT_LORA_RUN', '0') == '1':
    resume = None
print('Resuming from:', resume)
trainer.train(resume_from_checkpoint=resume)
post_train_lora_stats = lora_b_nonzero_stats(model)
print('Post-train LoRA B stats:', post_train_lora_stats)
if post_train_lora_stats['text_lora_B_nonzero_tensors'] == 0:
    raise RuntimeError('Training produced zero nonzero text lora_B tensors; adapter would not affect text generation.')
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
record_event('6_training', 'adapter_saved', adapter=str(ADAPTER_DIR), latest_checkpoint=latest_checkpoint(CKPT_DIR))
mark_phase_done('7_adapter_saved', [str(ADAPTER_DIR)], '8_merge')


# Free trainer/model memory before merge and generation eval.
import gc
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Released training model memory before merge.')


Forced transformers to recognize bitsandbytes.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

ValueError: Target module Gemma4ClippableLinear(
  (linear): Linear4bit(in_features=768, out_features=768, bias=False)
) is not supported. Currently, only the following modules are supported: `torch.nn.Linear`, `torch.nn.Embedding`, `torch.nn.Conv1d`, `torch.nn.Conv2d`, `torch.nn.Conv3d`, `transformers.pytorch_utils.Conv1D`, `torch.nn.MultiheadAttention.`.

In [ ]:
# Phase 8: Merge LoRA adapter into HF/safetensors export
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from packaging.version import Version

import torch

def clear_import_cache(prefixes: tuple[str, ...]) -> None:
    for name in list(sys.modules):
        if name in prefixes or any(name.startswith(prefix + '.') for prefix in prefixes):
            del sys.modules[name]
    importlib.invalidate_caches()

def ensure_gemma4_merge_deps() -> None:
    try:
        transformers_version = Version(importlib.metadata.version('transformers'))
    except importlib.metadata.PackageNotFoundError:
        transformers_version = Version('0')
    try:
        peft_version = Version(importlib.metadata.version('peft'))
    except importlib.metadata.PackageNotFoundError:
        peft_version = Version('0')
    if transformers_version < Version('5.5.0') or peft_version < Version('0.18.0'):
        print(f'Installing Gemma4 merge deps: transformers={transformers_version}, peft={peft_version}')
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '-U', '--pre',
            'transformers>=5.5.0', 'peft>=0.18.0', 'accelerate',
            'safetensors', 'huggingface_hub', 'packaging'
        ], check=True)
        clear_import_cache(('transformers', 'peft', 'accelerate', 'huggingface_hub', 'safetensors'))
    try:
        from transformers import AutoConfig
        model_id = os.environ.get('BASE_MODEL_ID', globals().get('BASE_MODEL_ID', 'google/gemma-4-E2B-it'))
        AutoConfig.from_pretrained(model_id, token=os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or globals().get('HF_TOKEN'))
    except Exception as exc:
        if 'gemma4' not in str(exc).lower() and 'model type' not in str(exc).lower():
            raise
        print('Current Transformers still does not recognize Gemma4; installing latest pre-release Transformers.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'transformers'], check=True)
        clear_import_cache(('transformers', 'peft'))

ensure_gemma4_merge_deps()
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

def _now_iso() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def _atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    tmp.replace(path)

def bootstrap_merge_context_if_needed() -> None:
    global HF_TOKEN, BASE_MODEL_ID, MODEL_TAG, RUN_SUFFIX, RUN_NAME
    global DRIVE_ROOT, TRAINED, ADAPTER_DIR, MERGED_DIR, PHASE_DIR, RUN_STATE, RUN_EVENTS, tokenizer
    if 'BASE_MODEL_ID' in globals() and 'ADAPTER_DIR' in globals() and 'MERGED_DIR' in globals():
        ADAPTER_DIR = Path(ADAPTER_DIR)
        MERGED_DIR = Path(MERGED_DIR)
        if 'tokenizer' not in globals():
            tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR) if ADAPTER_DIR.exists() else BASE_MODEL_ID, token=globals().get('HF_TOKEN'))
        return
    if Path('/content').exists():
        try:
            from google.colab import drive  # type: ignore
            if not Path('/content/drive/MyDrive').exists():
                drive.mount('/content/drive')
        except Exception as exc:
            print('Drive mount skipped/failed:', exc)
    HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
    BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
    DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_2_hard')
    USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'v5_2', 'v5_2_hard', 'hard'}
    RUN_SUFFIX = ('gemmafit_v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router')
    RUN_SUFFIX = os.environ.get('RUN_SUFFIX_OVERRIDE', RUN_SUFFIX)
    MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
    RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
    DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
    TRAINED = DRIVE_ROOT / 'trained_outputs'
    ADAPTER_DIR = TRAINED / 'adapter' / RUN_NAME
    MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
    PHASE_DIR = TRAINED / f'PHASE_DONE_{RUN_NAME}'
    RUN_STATE = TRAINED / f'RUN_STATE_{RUN_NAME}.json'
    RUN_EVENTS = TRAINED / f'RUN_EVENTS_{RUN_NAME}.jsonl'
    for path in [TRAINED, MERGED_DIR, PHASE_DIR]:
        path.mkdir(parents=True, exist_ok=True)
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f'Adapter not found: {ADAPTER_DIR}. Run/finish Phase 5/6 training first, or set DATASET_VARIANT/BASE_MODEL_ID to match the trained run.')
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), token=HF_TOKEN)
    print('Bootstrapped merge context:')
    print('  Base model :', BASE_MODEL_ID)
    print('  Adapter    :', ADAPTER_DIR)
    print('  Merged out :', MERGED_DIR)

bootstrap_merge_context_if_needed()

if 'record_event' not in globals():
    def record_event(phase: str, event: str, **payload) -> None:
        entry = {'time': _now_iso(), 'phase': phase, 'event': event, **payload}
        RUN_EVENTS.parent.mkdir(parents=True, exist_ok=True)
        with RUN_EVENTS.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        _atomic_json(RUN_STATE, entry)

if 'mark_phase_done' not in globals():
    def mark_phase_done(name: str, artifacts=None, next_phase: str = '') -> None:
        payload = {'time': _now_iso(), 'phase': name, 'artifacts': artifacts or [], 'next_phase': next_phase}
        _atomic_json(PHASE_DIR / f'{name}.json', payload)
        record_event(name, 'phase_done', artifacts=artifacts or [], next_phase=next_phase)

def ensure_torchao_for_peft_merge() -> None:
    minimum = Version('0.16.0')
    try:
        current = Version(importlib.metadata.version('torchao'))
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current is None or current < minimum:
        print(f'torchao {current or "not installed"} is incompatible with PEFT merge; installing torchao>=0.16.0')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', 'torchao>=0.16.0'], check=True)
        importlib.invalidate_caches()
        for name in list(sys.modules):
            if name == 'torchao' or name.startswith('torchao.'):
                del sys.modules[name]
    try:
        print('torchao version for PEFT merge:', importlib.metadata.version('torchao'))
    except importlib.metadata.PackageNotFoundError:
        print('torchao still unavailable; PEFT fallback will treat it as optional.')

def suppress_optional_torchao_in_peft() -> None:
    def _not_available(*args, **kwargs):
        return False
    for module in list(sys.modules.values()):
        if getattr(module, '__name__', '').startswith('peft') and hasattr(module, 'is_torchao_available'):
            setattr(module, 'is_torchao_available', _not_available)

def load_adapter_for_merge(base_model):
    try:
        return PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
    except ImportError as exc:
        if 'torchao' not in str(exc).lower():
            raise
        print('PEFT torchao compatibility check still failed; retrying with torchao marked unavailable for LoRA merge.')
        suppress_optional_torchao_in_peft()
        return PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

ensure_torchao_for_peft_merge()
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto', trust_remote_code=False)
merged = load_adapter_for_merge(base)
merged = merged.merge_and_unload()
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
record_event('8_merge', 'merged_hf_saved', merged_hf=str(MERGED_DIR))
mark_phase_done('8_merged_hf_saved', [str(MERGED_DIR)], '9_generation_smoke')


# Free merged model memory before generation eval subprocesses reload the saved model.
import gc
try:
    del base
except NameError:
    pass
try:
    del merged
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Released merge model memory after saving HF export.')


In [ ]:
# Phase 9: Generation smoke on held-out rows
import json
import os
import re
from pathlib import Path
from transformers import pipeline

def bootstrap_smoke_context_if_needed() -> None:
    global BASE_MODEL_ID, MODEL_TAG, RUN_SUFFIX, RUN_NAME, DRIVE_ROOT, TRAINED, MERGED_DIR, DATASET_PATH, tokenizer
    if 'MERGED_DIR' not in globals():
        BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
        DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_2_hard')
        USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'v5_2', 'v5_2_hard', 'hard'}
        RUN_SUFFIX = ('gemmafit_v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router')
        RUN_SUFFIX = os.environ.get('RUN_SUFFIX_OVERRIDE', RUN_SUFFIX)
        MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
        RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
        DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
        TRAINED = DRIVE_ROOT / 'trained_outputs'
        MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
    else:
        MERGED_DIR = Path(MERGED_DIR)
    if 'DATASET_PATH' not in globals():
        repo_dir = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
        DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_2_hard')
        dataset_name = 'gemmafit_v5_2_e2b_evidence_router.json' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router.json' if DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'hard'} else 'gemmafit_v5_e2b_evidence_router.json'
        DATASET_PATH = repo_dir / 'finetune' / 'data' / dataset_name
    if not MERGED_DIR.exists():
        raise FileNotFoundError(f'Merged model not found: {MERGED_DIR}. Run Phase 8 merge first.')
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}. Run Phase 3 dataset cell first or checkout repo data.')
    if 'tokenizer' not in globals():
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(str(MERGED_DIR))
    print('Smoke context:')
    print('  Merged model:', MERGED_DIR)
    print('  Dataset     :', DATASET_PATH)

def render_messages_for_smoke(row: dict) -> str:
    messages = row.get('messages', [])
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return '\n'.join(f"{m['role']}: {m['content']}" for m in messages)

bootstrap_smoke_context_if_needed()

if 'eval_pool' not in globals():
    with Path(DATASET_PATH).open('r', encoding='utf-8') as f:
        smoke_dataset = json.load(f)
    eval_pool = []
    for split_rows in smoke_dataset.get('validation', {}).values():
        eval_pool.extend(split_rows)
    print('Loaded eval rows for smoke:', len(eval_pool))

if 'record_event' not in globals():
    def record_event(phase: str, event: str, **payload) -> None:
        print('event', phase, event, payload)

if 'mark_phase_done' not in globals():
    def mark_phase_done(name: str, artifacts=None, next_phase: str = '') -> None:
        print('phase done', name, artifacts or [], next_phase)

SMOKE_LIMIT = int(os.environ.get('SMOKE_LIMIT', '8'))
text_gen = pipeline('text-generation', model=str(MERGED_DIR), tokenizer=tokenizer, device_map='auto')
smoke_rows = eval_pool[:SMOKE_LIMIT]
smoke_results = []
for idx, row in enumerate(smoke_rows):
    prompt_messages = row['messages'][:-1]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = '\n'.join(f"{m['role']}: {m['content']}" for m in prompt_messages) + '\nassistant:'
    out = text_gen(prompt, max_new_tokens=int(os.environ.get('MODEL_EVAL_MAX_NEW_TOKENS', '768')), do_sample=False, return_full_text=False)[0]['generated_text']
    start, end = out.find('{'), out.rfind('}')
    parsed = None
    if 0 <= start < end:
        try:
            parsed = json.loads(out[start:end + 1])
        except Exception:
            parsed = None
    smoke_results.append({'index': idx, 'row_type': row.get('row_type'), 'expected': row.get('expected_function'), 'parsed': parsed, 'raw': out[-1000:]})
print(json.dumps(smoke_results[:3], indent=2, ensure_ascii=False))
record_event('9_smoke', 'generation_smoke_done', rows=len(smoke_results))
mark_phase_done('9_generation_smoke_done', [], '10_handoff')


# Free smoke pipeline memory before formal generation eval.
import gc
try:
    del text_gen
except NameError:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
print('Released smoke generation pipeline memory.')


In [ ]:
# Phase 10: Optional LiteRT conversion / artifact handoff note
import json
import os
import time
from pathlib import Path

# Official LiteRT conversion is intentionally a separate step because package
# compatibility can differ from the training runtime. Use the conversion-only
# notebook or a clean runtime after MERGED_DIR exists.

def _handoff_now_iso() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def _handoff_atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    tmp.replace(path)

def bootstrap_handoff_context_if_needed() -> None:
    global BASE_MODEL_ID, MODEL_TAG, RUN_SUFFIX, RUN_NAME, DRIVE_ROOT, TRAINED, MERGED_DIR, PHASE_DIR, RUN_STATE, RUN_EVENTS
    if 'DRIVE_ROOT' not in globals():
        DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
    else:
        DRIVE_ROOT = Path(DRIVE_ROOT)
    if 'MERGED_DIR' not in globals():
        BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
        DATASET_VARIANT = os.environ.get('DATASET_VARIANT', 'v5_2_hard')
        USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'v5_2', 'v5_2_hard', 'hard'}
        RUN_SUFFIX = ('gemmafit_v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router')
        RUN_SUFFIX = os.environ.get('RUN_SUFFIX_OVERRIDE', RUN_SUFFIX)
        MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
        RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
        TRAINED = DRIVE_ROOT / 'trained_outputs'
        MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
        PHASE_DIR = TRAINED / f'PHASE_DONE_{RUN_NAME}'
        RUN_STATE = TRAINED / f'RUN_STATE_{RUN_NAME}.json'
        RUN_EVENTS = TRAINED / f'RUN_EVENTS_{RUN_NAME}.jsonl'
    else:
        MERGED_DIR = Path(MERGED_DIR)
    if not MERGED_DIR.exists():
        raise FileNotFoundError(f'Merged model not found: {MERGED_DIR}. Run Phase 8 merge first.')
    PHASE_DIR.mkdir(parents=True, exist_ok=True)

bootstrap_handoff_context_if_needed()

if 'record_event' not in globals():
    def record_event(phase: str, event: str, **payload) -> None:
        entry = {'time': _handoff_now_iso(), 'phase': phase, 'event': event, **payload}
        RUN_EVENTS.parent.mkdir(parents=True, exist_ok=True)
        with RUN_EVENTS.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        _handoff_atomic_json(RUN_STATE, entry)

if 'mark_phase_done' not in globals():
    def mark_phase_done(name: str, artifacts=None, next_phase: str = '') -> None:
        payload = {'time': _handoff_now_iso(), 'phase': name, 'artifacts': artifacts or [], 'next_phase': next_phase}
        _handoff_atomic_json(PHASE_DIR / f'{name}.json', payload)
        record_event(name, 'phase_done', artifacts=artifacts or [], next_phase=next_phase)

LITERT_TARGET = DRIVE_ROOT / 'gemmafit-v5-e2b-evidence-router.litertlm'
print('Merged HF export for conversion:', MERGED_DIR)
print('Expected final .litertlm    :', LITERT_TARGET)
print('Local finalize command after download:')
print('python finetune/prepare_litert_artifact.py --profile v5-e2b --source-litertlm path/to/gemmafit-v5-e2b-evidence-router.litertlm --run-smoke')
mark_phase_done('10_handoff_ready', [str(MERGED_DIR), str(LITERT_TARGET)], '11_done')


In [ ]:
# Phase 11: Eval reports and done marker
import hashlib
from collections import Counter
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

def _done_now_iso() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def _done_atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    tmp.replace(path)

def _latest_checkpoint(path: Path):
    path = Path(path)
    if not path.exists():
        return None
    checkpoints = sorted(path.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1)
    return str(checkpoints[-1]) if checkpoints else None

def bootstrap_done_context_if_needed() -> None:
    global BASE_MODEL_ID, MODEL_TAG, DATASET_VARIANT, USE_HARD_CASES, RUN_SUFFIX, RUN_NAME
    global DRIVE_ROOT, REPO_DIR, TRAINED, CKPT_DIR, ADAPTER_DIR, MERGED_DIR, PHASE_DIR, RUN_STATE, RUN_EVENTS, DISCONNECT_POINTS, DONE_MARKER
    global DATASET_REPO_PATH, DATASET_PATH, METRICS_DIR, TOOL_EVAL, REFUSAL_EVAL, ADVERSARIAL_EVAL
    BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', globals().get('BASE_MODEL_ID', 'google/gemma-4-E2B-it'))
    DATASET_VARIANT = os.environ.get('DATASET_VARIANT', globals().get('DATASET_VARIANT', 'v5_1_hard'))
    USE_HARD_CASES = DATASET_VARIANT in {'v5_1', 'v5_1_hard', 'v5_2', 'v5_2_hard', 'hard'}
    RUN_SUFFIX = ('gemmafit_v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'gemmafit_v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'gemmafit_v5_e2b_evidence_router')
    RUN_SUFFIX = os.environ.get('RUN_SUFFIX_OVERRIDE', RUN_SUFFIX)
    MODEL_TAG = BASE_MODEL_ID.replace('/', '_').replace('-', '_')
    RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'
    DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', globals().get('DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train')))
    REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', globals().get('REPO_DIR', '/content/GemmaFit')))
    TRAINED = DRIVE_ROOT / 'trained_outputs'
    CKPT_DIR = TRAINED / 'checkpoints' / RUN_NAME
    ADAPTER_DIR = TRAINED / 'adapter' / RUN_NAME
    MERGED_DIR = TRAINED / 'merged_fp16' / RUN_NAME
    PHASE_DIR = TRAINED / f'PHASE_DONE_{RUN_NAME}'
    RUN_STATE = TRAINED / f'RUN_STATE_{RUN_NAME}.json'
    RUN_EVENTS = TRAINED / f'RUN_EVENTS_{RUN_NAME}.jsonl'
    DISCONNECT_POINTS = TRAINED / f'DISCONNECT_POINTS_{RUN_NAME}.jsonl'
    DONE_MARKER = TRAINED / f'TRAINING_DONE_{RUN_NAME}.json'
    DATASET_REPO_PATH = Path('finetune/data/gemmafit_v5_2_e2b_evidence_router.json' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'finetune/data/gemmafit_v5_1_e2b_evidence_router.json' if USE_HARD_CASES else 'finetune/data/gemmafit_v5_e2b_evidence_router.json')
    DATASET_PATH = REPO_DIR / DATASET_REPO_PATH
    METRICS_DIR = REPO_DIR / 'finetune' / 'metrics'
    TOOL_EVAL = METRICS_DIR / 'tool_call_eval_v5_e2b.json'
    REFUSAL_EVAL = METRICS_DIR / 'refusal_eval_v5_e2b.json'
    ADVERSARIAL_EVAL = METRICS_DIR / 'adversarial_eval_v5_e2b.json'
    for path in [TRAINED, PHASE_DIR, METRICS_DIR]:
        path.mkdir(parents=True, exist_ok=True)
    if not DATASET_PATH.exists():
        raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}. Run Phase 3 dataset cell first.')
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f'Adapter not found: {ADAPTER_DIR}. Run/finish Phase 5/6 training first.')
    if not MERGED_DIR.exists():
        raise FileNotFoundError(f'Merged model not found: {MERGED_DIR}. Run Phase 8 merge first.')

bootstrap_done_context_if_needed()

if 'record_event' not in globals():
    def record_event(phase: str, event: str, **payload) -> None:
        entry = {'time': _done_now_iso(), 'phase': phase, 'event': event, **payload}
        RUN_EVENTS.parent.mkdir(parents=True, exist_ok=True)
        with RUN_EVENTS.open('a', encoding='utf-8') as f:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
        _done_atomic_json(RUN_STATE, entry)

if 'mark_phase_done' not in globals():
    def mark_phase_done(name: str, artifacts=None, next_phase: str = '') -> None:
        payload = {'time': _done_now_iso(), 'phase': name, 'artifacts': artifacts or [], 'next_phase': next_phase}
        _done_atomic_json(PHASE_DIR / f'{name}.json', payload)
        record_event(name, 'phase_done', artifacts=artifacts or [], next_phase=next_phase)

with DATASET_PATH.open('r', encoding='utf-8') as f:
    dataset_payload = json.load(f)
dataset_bytes = DATASET_PATH.read_bytes()
DATASET_SHA256 = hashlib.sha256(dataset_bytes).hexdigest()
TRAIN_ROWS = len(dataset_payload.get('train', []))
VALIDATION_ROWS = {k: len(v) for k, v in dataset_payload.get('validation', {}).items()}

# Deterministic dataset/schema eval. This is fast and catches generator contract drift.
subprocess.run([
    sys.executable,
    str(REPO_DIR / 'finetune' / 'eval_v5_e2b_evidence_router.py'),
    '--dataset', str(DATASET_PATH),
    '--output', str(TOOL_EVAL),
    '--refusal-output', str(REFUSAL_EVAL),
    '--adversarial-output', str(ADVERSARIAL_EVAL),
    '--strict',
], check=True)

# Phase 11b: Real generative tool-call eval. This is the acceptance signal; eval_loss is not enough.
def _truthy_env(name: str, default: str = '0') -> bool:
    return os.environ.get(name, default).strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def _load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def _normalize_parse_error(value: str) -> str:
    if not value:
        return 'none'
    value = re.sub(r'line \d+ column \d+.*', 'json_decode_error', value)
    value = re.sub(r'char \d+', 'char_n', value)
    return value[:160]

def _parsed_function(item: dict) -> str:
    parsed = item.get('parsed')
    if isinstance(parsed, dict):
        return str(parsed.get('function') or '<missing_function>')
    if item.get('parse_error'):
        return '<parse_failed>'
    return '<missing_parsed>'

def _write_generation_failure_analysis(output_json: Path, predictions_jsonl: Path) -> Path | None:
    if not output_json.exists():
        return None
    report = json.loads(output_json.read_text(encoding='utf-8'))
    predictions = _load_jsonl(predictions_jsonl)
    failures = report.get('failure_samples', [])
    issue_counts = Counter()
    missing_args = Counter()
    for item in failures:
        for issue in item.get('issues', []):
            issue_counts[issue] += 1
            if issue.startswith('missing_args:'):
                for arg in issue.split(':', 1)[1].split(','):
                    missing_args[arg] += 1
    parse_error_counts = Counter(_normalize_parse_error(item.get('parse_error', '')) for item in predictions if item.get('parse_error'))
    confusion = Counter((str(item.get('expected_function') or '<missing_expected>'), _parsed_function(item)) for item in predictions)
    row_type_failures = Counter(item.get('row_type', 'unknown') for item in failures)
    forbidden_samples = [item for item in failures if 'forbidden_claim' in item.get('issues', [])][:10]
    parse_fail_samples = [item for item in predictions if item.get('parse_error')][:10]
    schema_samples = [item for item in failures if any(str(issue).startswith('missing_args') for issue in item.get('issues', []))][:10]
    analysis = {
        'source_report': str(output_json),
        'source_predictions': str(predictions_jsonl),
        'summary': report.get('summary', {}),
        'generation': report.get('generation', {}),
        'gate_failures': report.get('gate_failures', []),
        'issue_counts': dict(issue_counts.most_common()),
        'missing_args': dict(missing_args.most_common()),
        'parse_error_counts': dict(parse_error_counts.most_common()),
        'function_confusion_top': [
            {'expected': expected, 'actual': actual, 'count': count}
            for (expected, actual), count in confusion.most_common(50)
        ],
        'row_type_failure_counts': dict(row_type_failures.most_common()),
        'row_type_summary': report.get('row_type_summary', {}),
        'forbidden_samples': forbidden_samples,
        'parse_fail_samples': parse_fail_samples,
        'schema_samples': schema_samples,
    }
    analysis_json = output_json.with_name(output_json.stem + '_failure_analysis.json')
    analysis_json.write_text(json.dumps(analysis, indent=2, ensure_ascii=False), encoding='utf-8')
    print('FAILURE_ANALYSIS:', analysis_json)
    print(json.dumps({
        'summary': analysis['summary'],
        'generation': analysis['generation'],
        'gate_failures': analysis['gate_failures'],
        'issue_counts': dict(issue_counts.most_common(25)),
        'missing_args': dict(missing_args.most_common(25)),
        'parse_error_counts': dict(parse_error_counts.most_common(10)),
        'function_confusion_top': analysis['function_confusion_top'][:20],
        'row_type_failure_counts': analysis['row_type_failure_counts'],
    }, indent=2, ensure_ascii=False))
    return analysis_json

RUN_GENERATION_EVAL_AFTER_TRAIN = _truthy_env('RUN_GENERATION_EVAL_AFTER_TRAIN', '1')
GENERATION_EVAL_REPORT = None
GENERATION_EVAL_PREDICTIONS = None
GENERATION_EVAL_FAILURE_ANALYSIS = None
GENERATION_EVAL_RETURN_CODE = None
GENERATION_EVAL_GATE_FAILURES = None
GENERATION_EVAL_STATUS = 'skipped'

if RUN_GENERATION_EVAL_AFTER_TRAIN:
    EVAL_DIR = TRAINED / 'evals'
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    RUN_NAME_SAFE = re.sub(r'[^A-Za-z0-9_.-]+', '_', RUN_NAME)
    GEN_EVAL_LIMIT = int(os.environ.get('GEN_EVAL_LIMIT', '320'))
    GEN_EVAL_SEED = int(os.environ.get('GEN_EVAL_SEED', '42'))
    GEN_EVAL_SPLITS = os.environ.get('GEN_EVAL_SPLITS', 'all')
    GEN_EVAL_ROW_TYPES = os.environ.get('GEN_EVAL_ROW_TYPES', 'all')
    GEN_EVAL_MAX_NEW_TOKENS = int(os.environ.get('GEN_EVAL_MAX_NEW_TOKENS', os.environ.get('MODEL_EVAL_MAX_NEW_TOKENS', '768')))
    GENERATION_EVAL_REPORT = Path(os.environ.get('GEN_EVAL_OUTPUT', str(EVAL_DIR / f'{RUN_NAME_SAFE}_eval_{GEN_EVAL_LIMIT}.json')))
    GENERATION_EVAL_PREDICTIONS = Path(os.environ.get('GEN_EVAL_PREDICTIONS', str(EVAL_DIR / f'{RUN_NAME_SAFE}_predictions_{GEN_EVAL_LIMIT}.jsonl')))
    generation_cmd = [
        sys.executable,
        str(REPO_DIR / 'finetune' / 'eval_v5_e2b_model_generation.py'),
        '--model', str(MERGED_DIR),
        '--dataset', str(DATASET_PATH),
        '--output', str(GENERATION_EVAL_REPORT),
        '--predictions-output', str(GENERATION_EVAL_PREDICTIONS),
        '--limit', str(GEN_EVAL_LIMIT),
        '--seed', str(GEN_EVAL_SEED),
        '--splits', GEN_EVAL_SPLITS,
        '--row-types', GEN_EVAL_ROW_TYPES,
        '--max-new-tokens', str(GEN_EVAL_MAX_NEW_TOKENS),
        '--device-map', os.environ.get('GEN_EVAL_DEVICE_MAP', 'auto'),
        '--dtype', os.environ.get('GEN_EVAL_DTYPE', 'float16'),
        '--sample-output-count', os.environ.get('GEN_EVAL_SAMPLE_OUTPUT_COUNT', '12'),
        '--progress-every', os.environ.get('GEN_EVAL_PROGRESS_EVERY', '10'),
    ]
    if _truthy_env('GEN_EVAL_LOAD_IN_4BIT', '0'):
        generation_cmd.append('--load-in-4bit')
    if _truthy_env('GEN_EVAL_STRICT', '0'):
        generation_cmd.append('--strict')
    print('Running generation eval:', ' '.join(generation_cmd))
    generation_result = subprocess.run(generation_cmd, check=False)
    GENERATION_EVAL_RETURN_CODE = generation_result.returncode
    if GENERATION_EVAL_REPORT.exists():
        generation_report = json.loads(GENERATION_EVAL_REPORT.read_text(encoding='utf-8'))
        GENERATION_EVAL_GATE_FAILURES = generation_report.get('gate_failures', [])
        GENERATION_EVAL_STATUS = 'passed' if not GENERATION_EVAL_GATE_FAILURES and GENERATION_EVAL_RETURN_CODE == 0 else 'completed_with_failures'
        GENERATION_EVAL_FAILURE_ANALYSIS = _write_generation_failure_analysis(GENERATION_EVAL_REPORT, GENERATION_EVAL_PREDICTIONS)
    else:
        GENERATION_EVAL_STATUS = 'failed_no_report'
    record_event(
        '11_generation_eval',
        'model_generation_eval_done',
        status=GENERATION_EVAL_STATUS,
        return_code=GENERATION_EVAL_RETURN_CODE,
        output=str(GENERATION_EVAL_REPORT) if GENERATION_EVAL_REPORT else None,
        failure_analysis=str(GENERATION_EVAL_FAILURE_ANALYSIS) if GENERATION_EVAL_FAILURE_ANALYSIS else None,
        gate_failures=GENERATION_EVAL_GATE_FAILURES,
    )
    if GENERATION_EVAL_RETURN_CODE not in (0, None) and _truthy_env('GEN_EVAL_STRICT', '0'):
        raise RuntimeError(f'Generation eval failed with return code {GENERATION_EVAL_RETURN_CODE}; see {GENERATION_EVAL_REPORT}')
    mark_phase_done(
        '11_generation_eval_done',
        [str(p) for p in [GENERATION_EVAL_REPORT, GENERATION_EVAL_PREDICTIONS, GENERATION_EVAL_FAILURE_ANALYSIS] if p],
        '11_done_marker',
    )

done = {
    'version': 'v5_2_e2b_evidence_router_tool_contract' if DATASET_VARIANT in {'v5_2', 'v5_2_hard'} else 'v5_1_e2b_evidence_router_hard_cases' if USE_HARD_CASES else 'v5_e2b_evidence_router',
    'run_name': RUN_NAME,
    'run_suffix': RUN_SUFFIX,
    'finished_at': _done_now_iso(),
    'status': 'training_complete',
    'last_completed_phase': '11_eval_done',
    'dataset_variant': DATASET_VARIANT,
    'hard_cases': USE_HARD_CASES,
    'dataset_path': str(DATASET_REPO_PATH),
    'dataset_sha256': DATASET_SHA256,
    'train_rows': TRAIN_ROWS,
    'validation_rows': VALIDATION_ROWS,
    'model': BASE_MODEL_ID,
    'checkpoint_dir': str(CKPT_DIR),
    'best_checkpoint': _latest_checkpoint(CKPT_DIR),
    'adapter_path': str(ADAPTER_DIR),
    'merged_hf_path': str(MERGED_DIR),
    'litertlm_path': 'models/gemmafit-v5-e2b-evidence-router.litertlm',
    'conversion_status': 'not_started',
    'tool_call_eval': str(TOOL_EVAL),
    'refusal_eval': str(REFUSAL_EVAL),
    'adversarial_eval': str(ADVERSARIAL_EVAL),
    'generation_eval_enabled': RUN_GENERATION_EVAL_AFTER_TRAIN,
    'generation_eval_status': GENERATION_EVAL_STATUS,
    'generation_eval_return_code': GENERATION_EVAL_RETURN_CODE,
    'generation_eval': str(GENERATION_EVAL_REPORT) if GENERATION_EVAL_REPORT else None,
    'generation_eval_predictions': str(GENERATION_EVAL_PREDICTIONS) if GENERATION_EVAL_PREDICTIONS else None,
    'generation_eval_failure_analysis': str(GENERATION_EVAL_FAILURE_ANALYSIS) if GENERATION_EVAL_FAILURE_ANALYSIS else None,
    'generation_eval_gate_failures': GENERATION_EVAL_GATE_FAILURES,
    'resume_log': str(RUN_EVENTS),
    'disconnect_points': str(DISCONNECT_POINTS),
}
_done_atomic_json(DONE_MARKER, done)
record_event('11_done', 'done_marker_written', done_marker=str(DONE_MARKER))
mark_phase_done('11_eval_done', [str(DONE_MARKER), str(TOOL_EVAL)], '')
print(json.dumps(done, indent=2, ensure_ascii=False))

if _truthy_env('AUTO_DISCONNECT_AFTER_DONE', '0'):
    print('AUTO_DISCONNECT_AFTER_DONE=1; unassigning Colab runtime after done marker and eval artifacts are written.')
    try:
        from google.colab import runtime  # type: ignore
        runtime.unassign()
    except Exception as exc:
        print('Runtime unassign skipped/failed:', exc)
